# Phase 4: Dashboard - Interactive Visualizations

Visual presentation of extreme weather events data in the USA (2000-2024).
The dashboard shows trends, most affected states, and event distribution.

In [1]:
## 1. Import Libraries

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded")

✓ Libraries loaded


In [2]:
## 2. Load Data

# Load the processed features dataset
df = pd.read_csv('../data/processed/features.csv')

print(f"✓ Data loaded: {df.shape[0]:,} records")
print(f"  Period: {df['YEAR'].min()}-{df['YEAR'].max()}")
print(f"  States: {df['STATE'].nunique()}")
print(f"  Event types: {df['EVENT_TYPE'].nunique()}")

✓ Data loaded: 1,491,990 records
  Period: 2000-2024
  States: 63
  Event types: 56


In [3]:
## 3. Chart 1: Severity Trend Over Time

# Aggregate average severity by year
severity_by_year = df.groupby('YEAR')['SEVERITY_INDEX'].mean().reset_index()

fig1 = px.line(
    severity_by_year,
    x='YEAR',
    y='SEVERITY_INDEX',
    title='Average Severity of Climate Events (2000-2024)',
    labels={'YEAR': 'Year', 'SEVERITY_INDEX': 'Severity Index'},
    markers=True,
    line_shape='linear'
)

fig1.update_layout(
    template='plotly_white',
    height=500,
    hovermode='x unified'
)

fig1.show()

In [4]:
## 4. Chart 2: Top 10 Most Affected States (Last 5 Years)

# Filter last 5 years
recent_years = df['YEAR'].max() - 4
df_recent = df[df['YEAR'] >= recent_years]

# Top 10 states by average severity
top_states = df_recent.groupby('STATE')['SEVERITY_INDEX'].mean().nlargest(10).reset_index()
top_states.columns = ['State', 'Average Severity']

fig2 = px.bar(
    top_states,
    x='State',
    y='Average Severity',
    title=f'Top 10 Most Affected States ({recent_years}-{df["YEAR"].max()})',
    labels={'State': 'State', 'Average Severity': 'Severity Index'},
    color='Average Severity',
    color_continuous_scale='Reds'
)

fig2.update_layout(
    template='plotly_white',
    height=500,
    xaxis_tickangle=-45
)

fig2.show()

In [5]:
## 5. Chart 3: Distribution of Event Types

# Count events by type
event_types = df['EVENT_TYPE'].value_counts().head(10).reset_index()
event_types.columns = ['Event Type', 'Count']

fig3 = px.bar(
    event_types,
    x='Event Type',
    y='Count',
    title='Top 10 Extreme Weather Event Types',
    labels={'Event Type': 'Event Type', 'Count': 'Number of Events'},
    color='Count',
    color_continuous_scale='Blues'
)

fig3.update_layout(
    template='plotly_white',
    height=500,
    xaxis_tickangle=-45
)

fig3.show()

In [6]:
## 6. Chart 4: Heatmap - States vs Event Types

# Create severity matrix: state x event type
# Use only top 15 states for readability
top_15_states = df_recent.groupby('STATE')['SEVERITY_INDEX'].mean().nlargest(15).index

df_heatmap = df[df['STATE'].isin(top_15_states)]
heatmap_data = df_heatmap.groupby(['STATE', 'EVENT_TYPE'])['SEVERITY_INDEX'].mean().reset_index()

# Filter to top 15 event types as well
top_15_events = df['EVENT_TYPE'].value_counts().head(15).index
heatmap_data = heatmap_data[heatmap_data['EVENT_TYPE'].isin(top_15_events)]

# Create pivot table
pivot_data = heatmap_data.pivot(index='STATE', columns='EVENT_TYPE', values='SEVERITY_INDEX')

fig4 = go.Figure(data=go.Heatmap(
    z=pivot_data.values,
    x=pivot_data.columns,
    y=pivot_data.index,
    colorscale='YlOrRd'
))

fig4.update_layout(
    title='Average Severity: States × Event Types',
    xaxis_title='Event Type',
    yaxis_title='State',
    height=600,
    xaxis_tickangle=-45
)

fig4.show()

In [7]:
## 7. Chart 5: Severity Evolution - Top 5 States

# Get top 5 states
top_5_states = df.groupby('STATE')['SEVERITY_INDEX'].mean().nlargest(5).index

# Data by year and state
df_top5 = df[df['STATE'].isin(top_5_states)].groupby(['YEAR', 'STATE'])['SEVERITY_INDEX'].mean().reset_index()

fig5 = px.line(
    df_top5,
    x='YEAR',
    y='SEVERITY_INDEX',
    color='STATE',
    title='Severity Trend - Top 5 States',
    labels={'YEAR': 'Year', 'SEVERITY_INDEX': 'Severity Index', 'STATE': 'State'},
    markers=True
)

fig5.update_layout(
    template='plotly_white',
    height=500,
    hovermode='x unified'
)

fig5.show()

In [8]:
## 8. Summary Statistics

print("="*70)
print("DASHBOARD SUMMARY - EXTREME WEATHER EVENTS IN THE USA")
print("="*70)

print(f"\n📊 GENERAL DATA:")
print(f"  Total events: {len(df):,}")
print(f"  Period analyzed: {df['YEAR'].min()}-{df['YEAR'].max()}")
print(f"  States involved: {df['STATE'].nunique()}")
print(f"  Event types: {df['EVENT_TYPE'].nunique()}")

print(f"\n⚠️  SEVERITY:")
print(f"  Average severity: {df['SEVERITY_INDEX'].mean():.4f}")
print(f"  Maximum severity: {df['SEVERITY_INDEX'].max():.4f}")
print(f"  Most affected state (average): {df.groupby('STATE')['SEVERITY_INDEX'].mean().idxmax()}")

print(f"\n💰 ECONOMIC IMPACT:")
total_damage = df['TOTAL_DAMAGE'].sum()
print(f"  Total damage (2000-2024): ${total_damage:,.0f}")

print(f"\n💔 LIVES LOST:")
total_deaths = df['TOTAL_DEATHS'].sum()
print(f"  Total deaths: {total_deaths:,.0f}")

print(f"\n🔥 MOST COMMON EVENTS:")
top_3_events = df['EVENT_TYPE'].value_counts().head(3)
for i, (event, count) in enumerate(top_3_events.items(), 1):
    print(f"  {i}. {event}: {count:,} events")

print("\n" + "="*70)
print("✓ Dashboard complete!")
print("="*70)

DASHBOARD SUMMARY - EXTREME WEATHER EVENTS IN THE USA

📊 GENERAL DATA:
  Total events: 1,491,990
  Period analyzed: 2000-2024
  States involved: 63
  Event types: 56

⚠️  SEVERITY:
  Average severity: 0.0555
  Maximum severity: 0.9590
  Most affected state (average): VERMONT

💰 ECONOMIC IMPACT:
  Total damage (2000-2024): $549,270,281,530

💔 LIVES LOST:
  Total deaths: 19,541

🔥 MOST COMMON EVENTS:
  1. Thunderstorm Wind: 384,590 events
  2. Hail: 286,401 events
  3. Flash Flood: 90,013 events

✓ Dashboard complete!
